In [ ]:
# ==========================================
# PART 3: DES ENCRYPTION TRACE (ROUND 1)
# ==========================================

# --- 1. CONSTANTS (From Lecture & Standard DES) ---
IP_TABLE = [
    58, 50, 42, 34, 26, 18, 10, 2, 60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6, 64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17, 9, 1, 59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5, 63, 55, 47, 39, 31, 23, 15, 7
]
PC1_TABLE = [
    57, 49, 41, 33, 25, 17, 9, 1, 58, 50, 42, 34, 26, 18,
    10, 2, 59, 51, 43, 35, 27, 19, 11, 3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15, 7, 62, 54, 46, 38, 30, 22,
    14, 6, 61, 53, 45, 37, 29, 21, 13, 5, 28, 20, 12, 4
]
PC2_TABLE = [
    14, 17, 11, 24, 1, 5, 3, 28, 15, 6, 21, 10,
    23, 19, 12, 4, 26, 8, 16, 7, 27, 20, 13, 2,
    41, 52, 31, 37, 47, 55, 30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53, 46, 42, 50, 36, 29, 32
]
E_BOX = [
    32, 1, 2, 3, 4, 5, 4, 5, 6, 7, 8, 9,
    8, 9, 10, 11, 12, 13, 12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21, 20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29, 28, 29, 30, 31, 32, 1
]
P_BOX = [
    16, 7, 20, 21, 29, 12, 28, 17, 1, 15, 23, 26, 5, 18, 31, 10,
    2, 8, 24, 14, 32, 27, 3, 9, 19, 13, 30, 6, 22, 11, 4, 25
]
# Standard DES S-Boxes (S1 through S8)
S_BOXES = [
    [[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7], [0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8], [4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0], [15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]],
    [[15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10], [3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5], [0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15], [13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]],
    [[10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8], [13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1], [13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7], [1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]],
    [[7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15], [13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9], [10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4], [3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]],
    [[2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9], [14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6], [4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14], [11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]],
    [[12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11], [10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8], [9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6], [4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]],
    [[4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1], [13,0,11,7,4,9,1,10,14,3,5,12,2,15,8,6], [1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2], [6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]],
    [[13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7], [1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2], [7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8], [2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]]
]

# --- 2. HELPER FUNCTIONS ---
def hex_to_bin(h): return bin(int(h, 16))[2:].zfill(len(h)*4)
def bin_to_hex(b): return hex(int(b, 2))[2:].upper().zfill(len(b)//4)
def permute(b, table): return "".join(b[i-1] for i in table)
def xor(b1, b2): return "".join('0' if x == y else '1' for x, y in zip(b1, b2))
def shift(b, n): return b[n:] + b[:n]

# --- 3. MAIN TRACE LOGIC ---
# Test Inputs
P_hex = "0123456789ABCDEF"
K_hex = "133457799BBCDFF1"

print(f"INPUTS:\nPlaintext: {P_hex}\nKey:       {K_hex}\n" + "-"*30)

# STEP A: Initial Permutation (IP)
P_bin = hex_to_bin(P_hex)
IP_out = permute(P_bin, IP_TABLE)
L0, R0 = IP_out[:32], IP_out[32:]

print(f"STEP A: Initial Permutation\nL0: {bin_to_hex(L0)}\nR0: {bin_to_hex(R0)}\n")

# STEP B: Key Schedule (Round 1)
K_bin = hex_to_bin(K_hex)
K_56 = permute(K_bin, PC1_TABLE)      # PC-1
C0, D0 = K_56[:28], K_56[28:]         # Split
C1, D1 = shift(C0, 1), shift(D0, 1)   # Left Shift (1 bit for Round 1)
K1 = permute(C1 + D1, PC2_TABLE)      # PC-2

print(f"STEP B: Subkey Generation\nK1: {bin_to_hex(K1)}\n")

# STEP C: The f-Function
R0_exp = permute(R0, E_BOX)           # Expansion
S_in = xor(R0_exp, K1)                # XOR with K1
S_out = ""

# S-Box Substitution Loop
for i in range(8):
    chunk = S_in[i*6 : (i+1)*6]       # Get 6 bits
    r = int(chunk[0] + chunk[5], 2)   # Row = bits 1 and 6
    c = int(chunk[1:5], 2)            # Col = bits 2,3,4,5
    val = S_BOXES[i][r][c]
    S_out += bin(val)[2:].zfill(4)

f_val = permute(S_out, P_BOX)         # P-Box Permutation

print(f"STEP C: f-Function Output\nf(R0, K1): {bin_to_hex(f_val)}\n")

# STEP D: Final XOR
L1 = R0                               # New Left is Old Right
R1 = xor(L0, f_val)                   # New Right is L0 XOR f(R0, K1)

print("-" * 30)
print(f"FINAL RESULT (ROUND 1):")
print(f"L1: {bin_to_hex(L1)}")
print(f"R1: {bin_to_hex(R1)}")

INPUTS:
Plaintext: 0123456789ABCDEF
Key:       133457799BBCDFF1
------------------------------
STEP A: Initial Permutation
L0: CC00CCFF
R0: F0AAF0AA

STEP B: Subkey Generation
K1: 1B02EFFC7072

STEP C: f-Function Output
f(R0, K1): 234AA9BB

------------------------------
FINAL RESULT (ROUND 1):
L1: F0AAF0AA
R1: EF4A6544
